In [3]:
import pandas as pd
import numpy as np

In [7]:
df.columns

Index(['timestamp', 'open', 'high', 'low', 'close', 'volume', 'previous_close',
       'tr1', 'tr2', 'tr3', 'true_range', 'atr_14', 'future_close',
       'future_return', 'target', 'log_return'],
      dtype='str')

In [15]:
df = pd.read_csv("../datas/btc_4h_full_features.csv")

df["ema_20"] = df["close"].ewm(span=20).mean()
df["ema_dist"] = (df["close"] - df["ema_20"]) / df["close"]

df["interaction"] = df["atr_14"] * df["ema_dist"]

df = df.dropna()

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

# Solo ATR
X = df[["atr_14"]]
y = df["target"]

split_index = int(len(df) * 0.7)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

y_proba = model.predict_proba(X_test_scaled)[:,1]

print("AUC ATR full dataset:", roc_auc_score(y_test, y_proba))

AUC ATR full dataset: 0.5532661604361371


In [17]:
df["vol_regime"] = pd.qcut(df["atr_14"], 3, labels=["low", "medium", "high"])

In [18]:
df_high = df[df["vol_regime"] == "high"]

X = df_high[["atr_14"]]
y = df_high["target"]

split_index = int(len(df_high) * 0.7)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

y_proba = model.predict_proba(X_test_scaled)[:,1]

print("AUC ATR in HIGH volatility regime:", roc_auc_score(y_test, y_proba))

AUC ATR in HIGH volatility regime: 0.578923724915186


In [19]:
# Target separati
df["target_long"] = (df["future_return"] > 0.025).astype(int)
df["target_short"] = (df["future_return"] < -0.025).astype(int)

print("Long frequency:", df["target_long"].mean())
print("Short frequency:", df["target_short"].mean())

Long frequency: 0.17733040638218128
Short frequency: 0.1529153504310114


In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

# LONG
X = df[["atr_14"]]
y = df["target_long"]

split_index = int(len(df) * 0.7)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

y_proba = model.predict_proba(X_test_scaled)[:,1]

print("AUC ATR for LONG:", roc_auc_score(y_test, y_proba))

AUC ATR for LONG: 0.515163354085274


In [21]:
# SHORT
X = df[["atr_14"]]
y = df["target_short"]

split_index = int(len(df) * 0.7)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

y_proba = model.predict_proba(X_test_scaled)[:,1]

print("AUC ATR for SHORT:", roc_auc_score(y_test, y_proba))

AUC ATR for SHORT: 0.5829231825047725


In [22]:
features = ["atr_14", "ema_dist", "interaction"]

X = df[features]
y = df["target_short"]

split_index = int(len(df) * 0.7)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

y_proba = model.predict_proba(X_test_scaled)[:,1]

print("AUC SHORT with interaction:", roc_auc_score(y_test, y_proba))

AUC SHORT with interaction: 0.5802088814641116


In [23]:
import numpy as np

# EMA 50
df["ema_50"] = df["close"].ewm(span=50).mean()

# Slope EMA
df["ema_slope"] = df["ema_50"].diff()

# ATR percentile (rolling lungo per regime)
df["atr_percentile"] = df["atr_14"].rolling(200).rank(pct=True)

# Volatility Z-score
df["volatility_z"] = (
    df["atr_14"] - df["atr_14"].rolling(200).mean()
) / df["atr_14"].rolling(200).std()

df = df.dropna()

df[["atr_percentile", "ema_slope", "volatility_z"]].describe()

,atr_percentile,ema_slope,volatility_z
count,18478.000000,18478.000000,18478.000000
mean,0.480016,3.387387,0.004226
std,0.313083,74.142104,1.240457
min,0.005000,-603.334049,-2.950897
25%,0.190000,-19.817059,-0.852684
50%,0.475000,1.227088,-0.236996
75%,0.755000,26.971558,0.596432
max,1.000000,492.317388,7.227185


In [24]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

regime_features = ["atr_percentile", "ema_slope", "volatility_z"]

X_regime = df[regime_features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_regime)

kmeans = KMeans(n_clusters=4, random_state=42)
df["regime"] = kmeans.fit_predict(X_scaled)

df["regime"].value_counts()

regime
1    8005
2    6432
0    2760
3    1281
Name: count, dtype: int64

In [25]:
df.groupby("regime")["target_short"].mean()

regime
0    0.153261
1    0.150281
2    0.141636
3    0.206089
Name: target_short, dtype: float64

In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

for r in sorted(df["regime"].unique()):
    
    df_r = df[df["regime"] == r]
    
    X = df_r[["atr_14"]]
    y = df_r["target_short"]
    
    split_index = int(len(df_r) * 0.7)
    
    X_train = X.iloc[:split_index]
    X_test = X.iloc[split_index:]
    y_train = y.iloc[:split_index]
    y_test = y.iloc[split_index:]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = LogisticRegression()
    model.fit(X_train_scaled, y_train)
    
    y_proba = model.predict_proba(X_test_scaled)[:,1]
    
    auc = roc_auc_score(y_test, y_proba)
    
    print(f"Regime {r} AUC:", round(auc, 4))

Regime 0 AUC: 0.6303
Regime 1 AUC: 0.536
Regime 2 AUC: 0.6441
Regime 3 AUC: 0.5286


In [27]:
df.groupby("regime")[["atr_percentile","ema_slope","volatility_z"]].mean()

,atr_percentile,ema_slope,volatility_z
regime,,,
0,0.881710,82.854034,1.729431
1,0.170179,4.537959,-0.997113
2,0.618025,-1.220847,0.185439
3,0.857763,-151.880454,1.634674


In [28]:
df["regime"].isin([0,2])

199      False
200      False
201      False
202      False
203      False
         ...  
18672    False
18673     True
18674     True
18675     True
18676     True
Name: regime, Length: 18478, dtype: bool

In [29]:
df_filtered = df[df["regime"].isin([0,2])]

X = df_filtered[["atr_14"]]
y = df_filtered["target_short"]

split_index = int(len(df_filtered) * 0.7)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

y_proba = model.predict_proba(X_test_scaled)[:,1]

print("Filtered Regime AUC:", roc_auc_score(y_test, y_proba))

Filtered Regime AUC: 0.6397806636100424


In [30]:
threshold = 0.7

y_pred_high = (y_proba > threshold).astype(int)

from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_high))
print("Predicted positives:", y_pred_high.sum())

              precision    recall  f1-score   support

           0       0.91      1.00      0.95      2501
           1       0.00      0.00      0.00       257

    accuracy                           0.91      2758
   macro avg       0.45      0.50      0.48      2758
weighted avg       0.82      0.91      0.86      2758

Predicted positives: 0


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

In [31]:
np.quantile(y_proba, [0.80, 0.85, 0.90, 0.95])

array([0.24430781, 0.2529706 , 0.26529064, 0.29068548])

In [32]:
threshold = np.quantile(y_proba, 0.90)

y_pred_top = (y_proba > threshold).astype(int)

print(classification_report(y_test, y_pred_top))
print("Predicted positives:", y_pred_top.sum())

              precision    recall  f1-score   support

           0       0.92      0.91      0.91      2501
           1       0.19      0.20      0.20       257

    accuracy                           0.84      2758
   macro avg       0.55      0.56      0.55      2758
weighted avg       0.85      0.84      0.85      2758

Predicted positives: 276


In [33]:
# Calcolo soglia tail (5% peggiore)
tail_threshold = df["future_return"].quantile(0.05)

print("Tail threshold:", tail_threshold)

df["tail_event"] = (df["future_return"] <= tail_threshold).astype(int)

print("Tail frequency:", df["tail_event"].mean())

Tail threshold: -0.05284005255496544
Tail frequency: 0.05000541184110834


In [34]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

X = df[["atr_14"]]
y = df["tail_event"]

split_index = int(len(df) * 0.7)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

y_proba = model.predict_proba(X_test_scaled)[:,1]

print("Global Tail AUC:", roc_auc_score(y_test, y_proba))

Global Tail AUC: 0.6611521611751561


In [35]:
for r in sorted(df["regime"].unique()):
    
    df_r = df[df["regime"] == r]
    
    X = df_r[["atr_14"]]
    y = df_r["tail_event"]
    
    split_index = int(len(df_r) * 0.7)
    
    X_train = X.iloc[:split_index]
    X_test = X.iloc[split_index:]
    y_train = y.iloc[:split_index]
    y_test = y.iloc[split_index:]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = LogisticRegression()
    model.fit(X_train_scaled, y_train)
    
    y_proba = model.predict_proba(X_test_scaled)[:,1]
    
    auc = roc_auc_score(y_test, y_proba)
    
    print(f"Regime {r} Tail AUC:", round(auc, 4))

Regime 0 Tail AUC: 0.5796
Regime 1 Tail AUC: 0.561
Regime 2 Tail AUC: 0.5684
Regime 3 Tail AUC: 0.528


In [36]:
threshold = np.quantile(y_proba, 0.90)

y_pred_top = (y_proba > threshold).astype(int)

from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_top))
print("Predicted positives:", y_pred_top.sum())
print("Tail events in test:", y_test.sum())

              precision    recall  f1-score   support

           0       0.95      0.89      0.92       366
           1       0.00      0.00      0.00        19

    accuracy                           0.85       385
   macro avg       0.47      0.45      0.46       385
weighted avg       0.90      0.85      0.87       385

Predicted positives: 39
Tail events in test: 19


In [37]:
df["year"] = pd.to_datetime(df["timestamp"]).dt.year
sorted(df["year"].unique())

[np.int32(2017),
 np.int32(2018),
 np.int32(2019),
 np.int32(2020),
 np.int32(2021),
 np.int32(2022),
 np.int32(2023),
 np.int32(2024),
 np.int32(2025),
 np.int32(2026)]

In [38]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import numpy as np

results = []

years = sorted(df["year"].unique())

for test_year in years[3:]:  # partiamo dopo avere almeno 3 anni di train
    
    train_data = df[df["year"] < test_year]
    test_data = df[df["year"] == test_year]
    
    if len(test_data) < 200:
        continue
    
    X_train = train_data[["atr_14"]]
    y_train = train_data["tail_event"]
    
    X_test = test_data[["atr_14"]]
    y_test = test_data["tail_event"]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = LogisticRegression()
    model.fit(X_train_scaled, y_train)
    
    y_proba = model.predict_proba(X_test_scaled)[:,1]
    
    auc = roc_auc_score(y_test, y_proba)
    
    results.append((test_year, auc, y_test.mean()))

for r in results:
    print(f"Year {r[0]} | AUC: {round(r[1],4)} | Tail freq: {round(r[2],4)}")

Year 2020 | AUC: 0.6305 | Tail freq: 0.0355
Year 2021 | AUC: 0.5934 | Tail freq: 0.0854
Year 2022 | AUC: 0.6263 | Tail freq: 0.0594
Year 2023 | AUC: 0.3381 | Tail freq: 0.0073
Year 2024 | AUC: 0.5945 | Tail freq: 0.0214
Year 2025 | AUC: 0.6144 | Tail freq: 0.0192
Year 2026 | AUC: 0.7754 | Tail freq: 0.0482


In [40]:
# Train su tutto il dataset storico tranne ultimo anno
train_data = df[df["year"] < 2025]
test_data = df[df["year"] == 2025]

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X_train = train_data[["atr_14"]]
y_train = train_data["tail_event"]

X_test = test_data[["atr_14"]]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

test_data["tail_score"] = model.predict_proba(X_test_scaled)[:,1]

test_data[["timestamp","tail_score"]].head()

,timestamp,tail_score
16134,2025-01-01 00:00:00,0.083956
16135,2025-01-01 04:00:00,0.082614
16136,2025-01-01 08:00:00,0.082516
16137,2025-01-01 12:00:00,0.083240
16138,2025-01-01 16:00:00,0.084485


In [41]:
test_data["tail_index"] = (
    test_data["tail_score"].rank(pct=True) * 100
)

test_data[["timestamp","tail_index"]].head()

,timestamp,tail_index
16134,2025-01-01 00:00:00,78.356164
16135,2025-01-01 04:00:00,76.392694
16136,2025-01-01 08:00:00,76.255708
16137,2025-01-01 12:00:00,77.442922
16138,2025-01-01 16:00:00,78.995434


In [43]:
def risk_level(x):
    if x > 95:
        return "Extreme"
    elif x > 85:
        return "High"
    elif x > 70:
        return "Elevated"
    else:
        return "Normal"

test_data["risk_level"] = test_data["tail_index"].apply(risk_level)

In [44]:
test_data[["timestamp", "tail_index", "risk_level"]].head()

,timestamp,tail_index,risk_level
16134,2025-01-01 00:00:00,78.356164,Elevated
16135,2025-01-01 04:00:00,76.392694,Elevated
16136,2025-01-01 08:00:00,76.255708,Elevated
16137,2025-01-01 12:00:00,77.442922,Elevated
16138,2025-01-01 16:00:00,78.995434,Elevated


In [45]:
extreme_data = test_data[test_data["risk_level"] == "Extreme"]

print("Extreme count:", len(extreme_data))
print("Tail freq in Extreme:", extreme_data["tail_event"].mean())

Extreme count: 110
Tail freq in Extreme: 0.02727272727272727


In [46]:
high_data = test_data[test_data["risk_level"] == "High"]

print("High count:", len(high_data))
print("Tail freq in High:", high_data["tail_event"].mean())

High count: 219
Tail freq in High: 0.0410958904109589


In [47]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import numpy as np

results = []

years = sorted(df["year"].unique())

for test_year in years[4:]:  # servono almeno 3 anni di storico
    
    train_years = [test_year-3, test_year-2, test_year-1]
    
    train_data = df[df["year"].isin(train_years)]
    test_data = df[df["year"] == test_year]
    
    if len(test_data) < 200:
        continue
    
    X_train = train_data[["atr_14"]]
    y_train = train_data["tail_event"]
    
    X_test = test_data[["atr_14"]]
    y_test = test_data["tail_event"]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = LogisticRegression()
    model.fit(X_train_scaled, y_train)
    
    y_proba = model.predict_proba(X_test_scaled)[:,1]
    
    auc = roc_auc_score(y_test, y_proba)
    
    # Top 10% concentration
    threshold = np.quantile(y_proba, 0.90)
    top_mask = y_proba > threshold
    concentration = y_test[top_mask].mean() if top_mask.sum() > 0 else 0
    
    results.append((test_year, auc, concentration, y_test.mean()))

for r in results:
    print(f"Year {r[0]} | AUC: {round(r[1],3)} | Top10% Tail Freq: {round(r[2],3)} | Baseline: {round(r[3],3)}")

Year 2021 | AUC: 0.593 | Top10% Tail Freq: 0.155 | Baseline: 0.085
Year 2022 | AUC: 0.626 | Top10% Tail Freq: 0.119 | Baseline: 0.059
Year 2023 | AUC: 0.338 | Top10% Tail Freq: 0.005 | Baseline: 0.007
Year 2024 | AUC: 0.595 | Top10% Tail Freq: 0.018 | Baseline: 0.021
Year 2025 | AUC: 0.614 | Top10% Tail Freq: 0.041 | Baseline: 0.019
Year 2026 | AUC: 0.775 | Top10% Tail Freq: 0.111 | Baseline: 0.048


In [48]:
# Rolling drawdown (20 periodi)
df["rolling_max_20"] = df["close"].rolling(20).max()
df["rolling_drawdown"] = (df["close"] - df["rolling_max_20"]) / df["rolling_max_20"]

# Rolling skewness e kurtosis (40 periodi su log_return)
df["rolling_skew"] = df["log_return"].rolling(40).skew()
df["rolling_kurtosis"] = df["log_return"].rolling(40).kurt()

df = df.dropna()

df[[
    "atr_percentile",
    "volatility_z",
    "ema_slope",
    "rolling_drawdown",
    "rolling_skew",
    "rolling_kurtosis"
]].describe()

,atr_percentile,volatility_z,ema_slope,rolling_drawdown,rolling_skew,rolling_kurtosis
count,18439.000000,18439.000000,18439.000000,18439.000000,18439.000000,18439.000000
mean,0.480771,0.006363,3.395358,-0.031786,-0.062965,3.299858
std,0.312945,1.240868,74.219613,0.037931,1.134435,3.743455
min,0.005000,-2.950897,-603.334049,-0.404837,-5.304504,-1.207956
25%,0.190000,-0.851073,-19.897574,-0.043303,-0.632936,0.880375
50%,0.475000,-0.234814,1.232967,-0.019177,-0.054198,2.179250
75%,0.755000,0.597957,27.115354,-0.006096,0.550559,4.493923
max,1.000000,7.227185,492.317388,0.000000,5.611378,33.840018


In [49]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import numpy as np

results_v2 = []

features = [
    "atr_percentile",
    "volatility_z",
    "ema_slope",
    "rolling_drawdown",
    "rolling_skew",
    "rolling_kurtosis"
]

years = sorted(df["year"].unique())

for test_year in years[4:]:
    
    train_years = [test_year-3, test_year-2, test_year-1]
    
    train_data = df[df["year"].isin(train_years)]
    test_data = df[df["year"] == test_year]
    
    if len(test_data) < 200:
        continue
    
    X_train = train_data[features]
    y_train = train_data["tail_event"]
    
    X_test = test_data[features]
    y_test = test_data["tail_event"]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_scaled, y_train)
    
    y_proba = model.predict_proba(X_test_scaled)[:,1]
    
    auc = roc_auc_score(y_test, y_proba)
    
    threshold = np.quantile(y_proba, 0.90)
    top_mask = y_proba > threshold
    concentration = y_test[top_mask].mean() if top_mask.sum() > 0 else 0
    
    results_v2.append((test_year, auc, concentration, y_test.mean()))

for r in results_v2:
    print(f"Year {r[0]} | AUC: {round(r[1],3)} | Top10% Tail Freq: {round(r[2],3)} | Baseline: {round(r[3],3)}")

Year 2021 | AUC: 0.461 | Top10% Tail Freq: 0.119 | Baseline: 0.085
Year 2022 | AUC: 0.571 | Top10% Tail Freq: 0.119 | Baseline: 0.059
Year 2023 | AUC: 0.49 | Top10% Tail Freq: 0.009 | Baseline: 0.007
Year 2024 | AUC: 0.494 | Top10% Tail Freq: 0.009 | Baseline: 0.021
Year 2025 | AUC: 0.611 | Top10% Tail Freq: 0.027 | Baseline: 0.019
Year 2026 | AUC: 0.659 | Top10% Tail Freq: 0.028 | Baseline: 0.048


In [50]:
df["atr_percentile"] = df["atr_14"].rolling(200).rank(pct=True)

df["volatility_z"] = (
    df["atr_14"] - df["atr_14"].rolling(200).mean()
) / df["atr_14"].rolling(200).std()

df["rolling_max_20"] = df["close"].rolling(20).max()
df["rolling_drawdown"] = (df["close"] - df["rolling_max_20"]) / df["rolling_max_20"]

df = df.dropna()

In [51]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import numpy as np

results_v3 = []

features = [
    "atr_14",
    "atr_percentile",
    "volatility_z",
    "rolling_drawdown"
]

years = sorted(df["year"].unique())

for test_year in years[4:]:
    
    train_years = [test_year-3, test_year-2, test_year-1]
    
    train_data = df[df["year"].isin(train_years)]
    test_data = df[df["year"] == test_year]
    
    if len(test_data) < 200:
        continue
    
    X_train = train_data[features]
    y_train = train_data["tail_event"]
    
    X_test = test_data[features]
    y_test = test_data["tail_event"]
    
    model = GradientBoostingClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    y_proba = model.predict_proba(X_test)[:,1]
    
    auc = roc_auc_score(y_test, y_proba)
    
    threshold = np.quantile(y_proba, 0.90)
    top_mask = y_proba > threshold
    concentration = y_test[top_mask].mean() if top_mask.sum() > 0 else 0
    
    results_v3.append((test_year, auc, concentration, y_test.mean()))

for r in results_v3:
    print(f"Year {r[0]} | AUC: {round(r[1],3)} | Top10% Tail Freq: {round(r[2],3)} | Baseline: {round(r[3],3)}")

Year 2021 | AUC: 0.486 | Top10% Tail Freq: 0.088 | Baseline: 0.085
Year 2022 | AUC: 0.568 | Top10% Tail Freq: 0.073 | Baseline: 0.059
Year 2023 | AUC: 0.626 | Top10% Tail Freq: 0.0 | Baseline: 0.007
Year 2024 | AUC: 0.612 | Top10% Tail Freq: 0.028 | Baseline: 0.021
Year 2025 | AUC: 0.344 | Top10% Tail Freq: 0.005 | Baseline: 0.019
Year 2026 | AUC: 0.667 | Top10% Tail Freq: 0.111 | Baseline: 0.048


In [53]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import numpy as np

df["tail_score_raw"] = np.nan

years = sorted(df["year"].unique())

for test_year in years[4:]:
    
    train_years = [test_year-3, test_year-2, test_year-1]
    
    train_data = df[df["year"].isin(train_years)]
    test_data = df[df["year"] == test_year]
    
    if len(test_data) < 200:
        continue
    
    X_train = train_data[["atr_14"]]
    y_train = train_data["tail_event"]
    
    X_test = test_data[["atr_14"]]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = LogisticRegression()
    model.fit(X_train_scaled, y_train)
    
    df.loc[df["year"] == test_year, "tail_score_raw"] = model.predict_proba(X_test_scaled)[:,1]

In [54]:
df["tail_score_smooth"] = df["tail_score_raw"].ewm(span=20).mean()

In [55]:
df["tail_index"] = (
    df["tail_score_smooth"]
    .rolling(365)
    .apply(lambda x: pd.Series(x).rank(pct=True).iloc[-1])
) * 100

In [56]:
df["tail_index"] = (
    df["tail_score_smooth"]
    .rolling(365)
    .apply(lambda x: pd.Series(x).rank(pct=True).iloc[-1])
) * 100

In [58]:
df.columns

Index(['timestamp', 'open', 'high', 'low', 'close', 'volume', 'previous_close',
       'tr1', 'tr2', 'tr3', 'true_range', 'atr_14', 'future_close',
       'future_return', 'target', 'log_return', 'ema_20', 'ema_dist',
       'interaction', 'vol_regime', 'target_long', 'target_short', 'ema_50',
       'ema_slope', 'atr_percentile', 'volatility_z', 'regime', 'tail_event',
       'year', 'rolling_max_20', 'rolling_drawdown', 'rolling_skew',
       'rolling_kurtosis', 'tail_score_raw', 'tail_score_smooth',
       'tail_index'],
      dtype='str')

In [59]:
def risk_level(x):
    if pd.isna(x):
        return None
    elif x > 95:
        return "Extreme"
    elif x > 85:
        return "High"
    elif x > 70:
        return "Elevated"
    else:
        return "Normal"

df["risk_level"] = df["tail_index"].apply(risk_level)

In [60]:
df["risk_level"].value_counts()

risk_level
Normal      7948
Elevated    1341
High        1039
Extreme      617
Name: count, dtype: int64

In [61]:
extreme = df[df["risk_level"] == "Extreme"]

print("Extreme count:", len(extreme))
print("Tail freq in Extreme:", extreme["tail_event"].mean())
print("Baseline tail freq:", df["tail_event"].mean())

Extreme count: 617
Tail freq in Extreme: 0.07779578606158834
Baseline tail freq: 0.05032894736842105
